# 字符级语言模型：RNN 文本生成 🦕

## 学习目标 🎯

完成本实验后，你将能够：

- 🔡 理解**字符级语言模型**的工作原理
- 📉 实现**梯度裁剪**防止梯度爆炸
- 🎲 从训练好的 RNN 中**采样生成**新序列
- 🔄 将前向传播、反向传播、裁剪和参数更新组合为完整的**优化步骤**
- 🦖 训练 RNN 模型生成类恐龙名字的新名称

## 目录

- [1 - 导入包](#1)
- [2 - 问题描述](#2)
  - [2.1 - 数据集与预处理](#2-1)
  - [2.2 - 模型结构概览](#2-2)
- [3 - 模型构建模块](#3)
  - [3.1 - 梯度裁剪](#3-1)
  - [3.2 - 字符采样](#3-2)
- [4 - 构建语言模型](#4)
  - [4.1 - 梯度下降优化步骤](#4-1)
  - [4.2 - 训练模型](#4-2)
- [5 - 总结](#5)

<a name='1'></a>
## 1 - 导入包 📦

运行下方代码块，加载本实验所需的全部依赖。

- **numpy**：数值计算核心库
- **utils**：本实验的辅助函数（RNN 前向/反向传播等）
- **random / copy**：随机采样与深拷贝

In [25]:
import numpy as np
from utils import *
import random
import pprint
import copy

<a name='2'></a>
## 2 - 问题描述 🦖

<a name='2-1'></a>
### 2.1 - 数据集与预处理

我们使用一个包含恐龙名字的数据集（`dinos.txt`）。模型将学习这些名字的字符规律，并尝试生成全新的"恐龙风格"名字。

运行下面的代码块读取数据集，并了解其基本信息。

In [26]:
data = open('dinos.txt', 'r').read()
data = data.lower()
chars = list(set(data))
data_size, vocab_size = len(data), len(chars)
print('数据集共有 %d 个字符，其中不重复字符 %d 个。' % (data_size, vocab_size))

数据集共有 19909 个字符，其中不重复字符 27 个。


数据集中包含的字符为 26 个英文字母（a-z）加上换行符 `\n`，共 **27** 个。

换行符 `\n` 在这里有特殊意义：
- 它标志着一个名字的**结束**
- 在训练时充当名字之间的**分隔符**

下面对字符集进行排序，并建立字符与整数索引之间的双向映射。

In [27]:
chars = sorted(chars)
print(chars)

['\n', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [28]:
char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}
pprint.pprint(ix_to_char)

{0: '\n',
 1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z'}


<a name='2-2'></a>
### 2.2 - 模型结构概览

本实验将训练一个循环神经网络（RNN），其结构如下：

$$a^{\langle t \rangle} = \tanh(W_{ax} x^{\langle t \rangle} + W_{aa} a^{\langle t-1 \rangle} + b)$$

$$\hat{y}^{\langle t \rangle} = \text{softmax}(W_{ya} a^{\langle t \rangle} + b_y)$$

**训练流程**：
1. 输入一个恐龙名字的字符序列
2. 模型预测每个位置的下一个字符
3. 使用交叉熵损失衡量预测准确度
4. 通过时间反向传播（BPTT）计算梯度
5. 裁剪梯度后更新参数

**推理/采样流程**：
- 从零向量（空起始符）开始
- 每次将上一步的输出作为下一步的输入
- 遇到换行符时停止

<a name='3'></a>
## 3 - 模型构建模块

<a name='3-1'></a>
### 3.1 - 梯度裁剪

在训练 RNN 时，梯度有时会变得非常大（**梯度爆炸**问题），导致参数更新过大、训练不稳定。

解决方法是**梯度裁剪**：将每个梯度值限制在 $[-\text{maxValue}, +\text{maxValue}]$ 范围内。

<a name='ex-1'></a>
### 练习 1 — `clip`

实现梯度裁剪函数。对字典 `gradients` 中的每个梯度数组，将其所有元素裁剪到 $[-\text{maxValue}, +\text{maxValue}]$ 范围内。

**提示**：使用 `np.clip(array, min_val, max_val, out=array)` 可以原地修改数组。

In [29]:
### GRADED FUNCTION: clip

def clip(gradients, maxValue):
    """
    将梯度字典中的所有梯度值裁剪到 [-maxValue, maxValue] 范围内。

    参数:
        gradients -- 梯度字典，包含键 'dWaa', 'dWax', 'dWya', 'db', 'dby'
        maxValue  -- 裁剪阈值，超过此值的梯度绝对值将被截断

    返回:
        gradients -- 裁剪后的梯度字典
    """
    gradients = copy.deepcopy(gradients)
    dWaa, dWax, dWya, db, dby = (gradients['dWaa'], gradients['dWax'],
                                  gradients['dWya'], gradients['db'], gradients['dby'])

    ### START CODE HERE ###
    ### END CODE HERE ###

    gradients = {"dWaa": dWaa, "dWax": dWax, "dWya": dWya, "db": db, "dby": dby}
    return gradients

In [30]:
# 梯度裁剪测试
np.random.seed(3)
dWax = np.random.randn(5, 3) * 10
dWaa = np.random.randn(5, 5) * 10
dWya = np.random.randn(2, 5) * 10
db = np.random.randn(5, 1) * 10
dby = np.random.randn(2, 1) * 10
gradients_test = {"dWax": dWax, "dWaa": dWaa, "dWya": dWya, "db": db, "dby": dby}

gradients_out = clip(gradients_test, 10)
assert gradients_out["dWaa"][1][2] == 10.0, "dWaa 裁剪失败"
assert gradients_out["dWax"][3][1] == -10.0, "dWax 裁剪失败"
assert gradients_out["dWya"][0][0] == -10.0, "dWya 裁剪失败"
print("\033[92mAll tests passed")

All tests passed


<a name='3-2'></a>
### 3.2 - 字符采样

训练完成后，可以从模型中**采样**生成新的字符序列。采样过程：

1. 以全零向量 $x^{\langle 1 \rangle}$ 作为起始输入（无先验字符）
2. 初始化隐藏状态 $a^{\langle 0 \rangle} = \mathbf{0}$
3. 循环执行前向传播，每步根据概率分布 $\hat{y}^{\langle t \rangle}$ 随机采样一个字符索引
4. 将采样到的字符的 one-hot 向量作为下一步输入
5. 遇到换行符 `\n` 时停止

<a name='ex-2'></a>
### 练习 2 — `sample`

实现字符采样函数。

In [31]:
### GRADED FUNCTION: sample

def sample(parameters, char_to_ix, seed):
    """
    从训练好的 RNN 字符模型中采样一个字符序列。

    参数:
        parameters -- 权重字典，包含 Waa, Wax, Wya, by, b
        char_to_ix -- 字符到索引的映射字典
        seed       -- 随机种子（用于结果复现）

    返回:
        indices -- 采样得到的字符索引列表
    """
    Waa, Wax, Wya, by, b = (parameters['Waa'], parameters['Wax'],
                              parameters['Wya'], parameters['by'], parameters['b'])
    vocab_size = by.shape[0]
    n_a = Waa.shape[1]

    ### START CODE HERE ###
    # 步骤 1：初始化输入向量 x 和隐藏状态 a_prev 为零向量


        # 步骤 2：执行一步前向传播


        # 步骤 3：根据概率分布采样字符索引

        # 步骤 4：将采样字符的 one-hot 向量作为下一步输入

    ### END CODE HERE ###

    if counter == 50:
        indices.append(char_to_ix['\n'])

    return indices

In [32]:
# 字符采样测试
np.random.seed(2)
_, n_a = 20, 100
Wax_t = np.random.randn(n_a, 20)
Waa_t = np.random.randn(n_a, n_a)
Wya_t = np.random.randn(20, n_a)
b_t = np.random.randn(n_a, 1)
by_t = np.random.randn(20, 1)
parameters_t = {"Wax": Wax_t, "Waa": Waa_t, "Wya": Wya_t, "b": b_t, "by": by_t}
char_to_ix_t = {chr(ord('a') + i): i for i in range(19)}
char_to_ix_t['\n'] = 19

indices_out = sample(parameters_t, char_to_ix_t, 0)
assert len(indices_out) > 0, "采样结果不应为空"
assert indices_out[-1] == 19 or len(indices_out) == 50, "采样应以换行符结尾或达到最大长度 50"
print("采样得到的索引列表:", indices_out)
print("\033[92mAll tests passed")

采样得到的索引列表: [17, 9, 17, 19]
All tests passed


<a name='4'></a>
## 4 - 构建语言模型

<a name='4-1'></a>
### 4.1 - 梯度下降优化步骤

现在将前向传播、反向传播、梯度裁剪和参数更新整合为一个完整的优化步骤。

<a name='ex-3'></a>
### 练习 3 — `optimize`

实现一步完整的优化过程：
1. 调用 `rnn_forward` 进行前向传播，获取损失和缓存
2. 调用 `rnn_backward` 进行反向传播，获取梯度
3. 调用 `clip` 对梯度进行裁剪（`maxValue=5`）
4. 调用 `update_parameters` 更新参数

In [33]:
### GRADED FUNCTION: optimize

def optimize(X, Y, a_prev, parameters, learning_rate=0.01):
    """
    执行一步完整的优化：前向传播 → 反向传播 → 梯度裁剪 → 参数更新。

    参数:
        X             -- 输入字符索引列表（第一个元素为 None）
        Y             -- 目标字符索引列表
        a_prev        -- 上一时刻的隐藏状态，形状 (n_a, 1)
        parameters    -- 当前参数字典
        learning_rate -- 学习率，默认 0.01

    返回:
        loss       -- 本步骤的交叉熵损失
        gradients  -- 裁剪后的梯度字典
        a[len(X)-1] -- 序列最后一个时刻的隐藏状态
    """
    ### START CODE HERE ###
    # 前向传播

    # 反向传播

    # 梯度裁剪

    # 参数更新
    ### END CODE HERE ###

    return loss, gradients, a[len(X) - 1]

In [34]:
# optimize 测试
np.random.seed(1)
vocab_size_t, n_a_t = 27, 100
a_prev_t = np.random.randn(n_a_t, 1)
parameters_t = initialize_parameters(n_a_t, vocab_size_t, vocab_size_t)
X_t = [12, 3, 5, 11, 22, 3]
Y_t = [4, 14, 11, 22, 25, 26]
loss_t, gradients_t, a_last_t = optimize(X_t, Y_t, a_prev_t, parameters_t, learning_rate=0.01)

assert loss_t > 0, "损失应为正数"
assert a_last_t.shape == (n_a_t, 1), "隐藏状态形状不正确"
assert gradients_t['dWaa'].shape == (n_a_t, n_a_t), "dWaa 形状不正确"
print("损失 =", round(loss_t, 4))
print("gradients['dWaa'][1][2] =", gradients_t['dWaa'][1][2])
print("\033[92mAll tests passed")

损失 = 19.7738
gradients['dWaa'][1][2] = 0.0026501576846759533
All tests passed


<a name='4-2'></a>
### 4.2 - 训练模型

最后，将所有组件整合为完整的训练循环。

**训练流程说明**：
- 每次迭代从训练集中循环选取一个恐龙名字
- 将名字转换为字符索引序列 `X`（首位补 `None`）和标签序列 `Y`（末位加换行符索引）
- 调用 `optimize` 执行一步优化
- 使用指数平滑跟踪损失变化
- 每 2000 次迭代打印当前损失和采样生成的名字

<a name='ex-4'></a>
### 练习 4 — `model`

完成 `model` 函数中标注的代码段。

In [35]:
### GRADED FUNCTION: model

def model(data_x, ix_to_char, char_to_ix, num_iterations=35000, n_a=50,
          dino_names=7, vocab_size=27, verbose=False):
    """
    训练字符级 RNN 语言模型，并周期性地采样生成恐龙名字。

    参数:
        data_x         -- 训练数据，每行一个恐龙名字（字符串列表）
        ix_to_char     -- 索引到字符的映射字典
        char_to_ix     -- 字符到索引的映射字典
        num_iterations -- 训练迭代总次数，默认 35000
        n_a            -- RNN 隐藏单元数，默认 50
        dino_names     -- 每次打印时采样的名字数量，默认 7
        vocab_size     -- 词表大小，默认 27
        verbose        -- 是否打印调试信息，默认 False

    返回:
        parameters -- 训练后的参数字典
        last_name  -- 最后一次采样得到的名字（用于测试）
    """
    n_x, n_y = vocab_size, vocab_size
    parameters = initialize_parameters(n_a, n_x, n_y)
    loss = get_initial_loss(vocab_size, dino_names)

    examples = [x.strip() for x in data_x]
    np.random.seed(0)
    np.random.shuffle(examples)

    a_prev = np.zeros((n_a, 1))
    last_dino_name = "abc"

    for j in range(num_iterations):

        ### START CODE HERE ###
        # 循环选取训练样本

        # 将当前名字转换为字符索引序列

        # 构建标签序列：输入向右移一位，末尾追加换行符

        # 执行一步优化（学习率 0.01）
        ### END CODE HERE ###

        if verbose and j in [0, len(examples) - 1, len(examples)]:
            print("j =", j, "idx =", idx)
        if verbose and j == 0:
            print("single_example =", single_example)
            print("single_example_ix =", single_example_ix)
            print("X =", X)
            print("Y =", Y)

        # 指数平滑损失
        loss = smooth(loss, curr_loss)

        # 每 2000 次迭代打印损失和采样结果
        if j % 2000 == 0:
            print('迭代次数: %d, 损失: %f' % (j, loss) + '\n')
            seed = 0
            for name in range(dino_names):
                sampled_indices = sample(parameters, char_to_ix, seed)
                last_dino_name = get_sample(sampled_indices, ix_to_char)
                print(last_dino_name.replace('\n', ''))
                seed += 1
            print('\n')

    return parameters, last_dino_name

运行下面的代码块开始训练模型。训练约需要几分钟。

模型训练初期会生成无意义的随机字符序列，随着迭代次数增加，生成的名字会逐渐变得像真实的恐龙名字。

In [36]:
parameters, last_name = model(data.split("\n"), ix_to_char, char_to_ix, 22001, verbose=True)

j = 0 idx = 0
single_example = turiasaurus
single_example_ix = [20, 21, 18, 9, 1, 19, 1, 21, 18, 21, 19]
X = [None, 20, 21, 18, 9, 1, 19, 1, 21, 18, 21, 19]
Y = [20, 21, 18, 9, 1, 19, 1, 21, 18, 21, 19, 0]
迭代次数: 0, 损失: 23.087336

Nkzxwtdmfqoeyhsqwasjkjvu
Kneb
Kzxwtdmfqoeyhsqwasjkjvu
Neb
Zxwtdmfqoeyhsqwasjkjvu
Eb
Xwtdmfqoeyhsqwasjkjvu


j = 1535 idx = 1535
j = 1536 idx = 0
迭代次数: 2000, 损失: 27.884160

Liusskeomnolxeros
Hmdaairus
Hytroligoraurus
Lecalosapaus
Xusicikoraurus
Abalpsamantisaurus
Tpraneronxeros


迭代次数: 4000, 损失: 25.901815

Mivrosaurus
Inee
Ivtroplisaurus
Mbaaisaurus
Wusichisaurus
Cabaselachus
Toraperlethosdarenitochusthiamamumamaon


迭代次数: 6000, 损失: 24.608779

Onwusceomosaurus
Lieeaerosaurus
Lxussaurus
Oma
Xusteonosaurus
Eeahosaurus
Toreonosaurus


迭代次数: 8000, 损失: 24.070350

Onxusichepriuon
Kilabersaurus
Lutrodon
Omaaerosaurus
Xutrcheps
Edaksoje
Trodiktonus


迭代次数: 10000, 损失: 23.844446

Onyusaurus
Klecalosaurus
Lustodon
Ola
Xusodonia
Eeaeosaurus
Troceosaurus


迭代次数: 12000, 损失: 

<a name='5'></a>
## 5 - 总结 🎉

### 本实验完成了：

✅ **梯度裁剪**：通过将梯度限制在 $[-5, 5]$ 范围内，有效防止了梯度爆炸问题

✅ **字符采样**：实现了从 RNN 概率分布中逐字符采样生成新序列的算法

✅ **优化步骤**：将前向传播、反向传播、梯度裁剪和参数更新整合为标准优化循环

✅ **模型训练**：训练了一个字符级语言模型，能够生成具有恐龙名字风格的新名称

### 核心知识点：

| 概念 | 说明 |
|------|------|
| 梯度爆炸 | RNN 训练中因时间步累积导致梯度过大的问题 |
| 梯度裁剪 | 将梯度元素值截断到指定范围的技术 |
| 字符级模型 | 以单个字符为最小处理单元的语言模型 |
| BPTT | 通过时间反向传播（Backpropagation Through Time） |
| 采样温度 | 控制生成多样性的超参数（本实验使用默认值） |

### 参考资料

- Karpathy, A. (2015). [The Unreasonable Effectiveness of Recurrent Neural Networks](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press.